# 🧠 AI Model 1: The Static Hand Geometry Classifier (CNN/Dense)
This notebook downloads the 50,000+ ASL Image Dataset from Kaggle, perfectly maps every image into a 1662-length MediaPipe Coordinate Array (blinding the AI to the face and body), and trains our ultra-fast Static router engine.

In [ ]:
# 1. SECURE LOGIN TO KAGGLE AND DOWNLOAD DATASET
import os
import zipfile

# ⚠️ IMPORTANT: I loaded your Secret Token! BUT YOU MUST TYPE YOUR EXACT KAGGLE USERNAME BELOW:
os.environ['KAGGLE_USERNAME'] = 'ENTER_YOUR_KAGGLE_USERNAME_HERE' 
os.environ['KAGGLE_KEY'] = 'KGAT_e7711c554483bb721b2b0adf866d3517'

!pip install kaggle mediapipe tensorflow opencv-python

# Force download massive dataset directly into Google Server!
!kaggle datasets download -d ahmedkhanak1995/sign-language-gesture-images-dataset

!unzip -q sign-language-gesture-images-dataset.zip -d dataset
print("✅ Download & Extraction Complete!")


In [ ]:
# 2. MATHEMATICAL MEDIAPIPE EXTRACTION
# This converts regular JPEG pictures into 1662 math coordinates!
import cv2
import mediapipe as mp
import numpy as np
import glob

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=True, min_detection_confidence=0.5)

def extract_keypoints(results):
    # We intentionally zero out Pose and Face just like in our React Component!
    pose = np.zeros(132)
    face = np.zeros(1404)
    lh = np.zeros(63)
    rh = np.zeros(63)
    if results.left_hand_landmarks:
        lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten()
    if results.right_hand_landmarks:
        rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten()
    return np.concatenate([pose, face, lh, rh])

X, Y = [], []

# Path to the unzipped images (Change path slightly if Kaggle's internal unzipped folder names differ)
folders = glob.glob('dataset/Gesture Image Data/*')
classes = [os.path.basename(f) for f in folders]
classes.sort()

print(f"Found classes: {classes}")
import json
with open('static_classes.json', 'w') as f:
    json.dump(classes, f)

for label, folder in enumerate(classes):
    print(f'Running Math Extraction on Class: {folder}...')
    images = glob.glob(folder + '/*.png') + glob.glob(folder + '/*.jpg')
    # Limiting to 300 perfectly tracked images per class to keep Google Colab RAM healthy
    valid_count = 0
    for img_path in images:
        if valid_count >= 300: break
        try:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            if results.left_hand_landmarks or results.right_hand_landmarks:
                keypoints = extract_keypoints(results)
                X.append(keypoints)
                Y.append(label)
                valid_count += 1
        except Exception:
            continue

X = np.array(X)
Y = np.array(Y)
print(f"✅ Successfully packaged {len(X)} mathematical arrays!")


In [ ]:
# 3. TRAIN THE SPATIAL NEURAL NETWORK
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

y_cat = to_categorical(Y, num_classes=len(classes))
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.1)

model = Sequential([
    Dense(256, activation='relu', input_shape=(1662,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(len(classes), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("🚀 IGNITING AI TRAINING ENGINE...")
model.fit(X_train, y_train, epochs=40, batch_size=32, validation_data=(X_test, y_test))

model.save('static_model.h5')
print('\n🟢 SUCCESS! You can now download static_model.h5 and static_classes.json from the Google Colab file menu on the left!')
